In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

IMG_SIZE = (224, 224)
BATCH_SIZE = 16 
EPOCHS = 20
BASE_PATH = "D:/VS Code/Coding/Deepfake_Project/Phase1_Preprocessing/processed_data"
CHECKPOINT_PATH = "D:/VS Code/Coding/Deepfake_Project/Phase2_Training/best_model.h5"

train_datagen = ImageDataGenerator(rescale=1./255, rotation_range=20, horizontal_flip=True, validation_split=0.2)

train_generator = train_datagen.flow_from_directory(BASE_PATH, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', subset='training')
val_generator = train_datagen.flow_from_directory(BASE_PATH, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', subset='validation')

# If power went out and want to RESUME, uncomment the line below:
# model = tf.keras.models.load_model(CHECKPOINT_PATH) 

def build_model():
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)
    predictions = Dense(1, activation='sigmoid')(x)
    return Model(inputs=base_model.input, outputs=predictions)

model = build_model() # Comment this out if you are resuming from a loaded model

model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

# --- 4. Callbacks ---
# Early Stopping: Stops if model stops improving
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)

# Model Checkpoint: Saves the "Best" version of the model automatically [cite: 49, 170]
checkpoint = ModelCheckpoint(
    filepath=CHECKPOINT_PATH,
    monitor='val_accuracy',
    save_best_only=True,   # Only save if accuracy improves
    mode='max',
    verbose=1
)

# --- 5. Training ---
print("🚀 Starting Phase 2 with Checkpointing enabled...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=[early_stop, checkpoint], # Both safety nets active
    verbose=1
)

# Final Save
model.save("D:/VS Code/Coding/Deepfake_Project/Phase2_Training/artifact_lens_final.h5")